# Capstone 02 — Multi-agent Research System with Provenance · Homework (self-assessment)

**Goal:** check that you can design and debug the research system from the [README](README.md): explicit context passing, bounded parallel fan-out, error envelopes computed in code, conflict rules, coverage annotations and an eval metric, then score your own build against the capstone rubric and defend it in two design reviews.

- Brief and rubric: [README.md](README.md) · Build: [01_practice.ipynb](01_practice.ipynb)
- **Estimated time:** 2–3 hours (plus the time to gather evidence for the rubric)
- **Estimated API cost:** **$0**. Everything graded runs offline; no key is needed.
- **Prerequisites:** the practice notebook, and the venv from [00-prerequisites](../../../00-prerequisites/README.md)

**How to use it**
1. Attempt everything **without peeking**. Solutions are at the very bottom, collapsed.
2. **Part A:** 10 original exam-style questions; fill in `answers`, then run the grader.
3. **Part B:** six auto-checked extensions. Replace each `TODO` stub, then run its checker. Checkers never crash the notebook; they tell you what is wrong.
4. **Part C:** score your build on the capstone rubric, with evidence.
5. **Part D:** two design reviews ("what would you change if…"); write your answer, then compare with the model answer.
6. Run the **Scorecard**. Pass mark: **72%** on quiz + exercises (the same bar as the Claude exam's 720/1000), plus **72%** on the rubric with no red flag.

The quiz items are original and written for this repo. They test the same competencies as the official guide's scenario, not its questions.

In [ ]:
import asyncio
import hashlib
import inspect
import json
import re
import time
from typing import Callable, Literal

from pydantic import BaseModel, Field, ValidationError, model_validator

print("ready")

### Helpers (run, don't edit)

The practice notebook's contracts (`Claim`, `Evidence`, `SubagentResult` with its validator, `Brief`), a small source registry (`P1` is a press release, `N1` a news article that only re-reports it, `L1` has no publication date), the claim builder `mk(...)`, `origins()`, the practice classifier `classify_pair_v1()`, `failed_envelope()`, and the text helpers `tag()`, `nums()`, `REF` and `split_sentences()`.

In [ ]:
# ---------- grading bookkeeping ----------
RESULTS: dict[int, bool] = {}
SCORE: dict[str, float] = {}


def _report(n: int, exc: BaseException | None) -> None:
    if exc is None:
        RESULTS[n] = True
        print(f"✓ Exercise {n} passed")
    elif isinstance(exc, NotImplementedError):
        RESULTS[n] = False
        print(f"✗ Exercise {n}: not implemented yet (the stub still raises NotImplementedError)")
    elif isinstance(exc, AssertionError):
        RESULTS[n] = False
        print(f"✗ Exercise {n}: {exc}")
    elif isinstance(exc, BaseExceptionGroup):
        RESULTS[n] = False
        print(f"✗ Exercise {n}: a TaskGroup re-raised a worker failure and cancelled the healthy workers ({exc}). "
              "Catch errors per worker and collect results with asyncio.gather")
    else:
        RESULTS[n] = False
        print(f"✗ Exercise {n}: your code raised {type(exc).__name__}: {exc}")


def _run_check(n: int, check: Callable[[], None]) -> None:
    """Run one checker; never let it crash the notebook."""
    try:
        check()
        _report(n, None)
    except BaseException as exc:  # noqa: BLE001 - report, don't crash
        _report(n, exc)


async def _run_check_async(n: int, check) -> None:
    try:
        await check()
        _report(n, None)
    except BaseException as exc:  # noqa: BLE001 - report, don't crash (includes ExceptionGroup)
        _report(n, exc)


# ---------- the capstone's contracts (same as the practice notebook) ----------
QUESTION = ("Prepare a research brief on Riverton's shared e-scooter and e-bike program (launched 2023) for the "
            "council's contract-renewal vote: ridership, safety, emissions, local economy and regulation.")
AS_OF = "2026-10-03"
REQUIRED_FACETS = ["ridership", "safety", "emissions", "economy", "regulation"]
Facet = Literal["ridership", "safety", "emissions", "economy", "regulation"]
FailureType = Literal["timeout", "rate_limited", "unavailable", "invalid_input", "permission_denied", "internal_error"]
CONTRACT = ("Answer only from what you receive. One short cited sentence per finding; keep qualifiers. "
            "Say which searches found nothing and which failed.")


class Source(BaseModel):
    source_id: str
    title: str
    published: str | None
    methodology: str | None
    derived_from: str | None


class Evidence(BaseModel):
    source_id: str
    excerpt: str
    location: str | None
    verbatim: bool


class Claim(BaseModel):
    claim_id: str
    facet: Facet
    statement: str
    subject: str
    metric: str | None
    value: float | None
    unit: str | None
    period: str | None
    evidence: list[Evidence] = Field(min_length=1)


class SubagentResult(BaseModel):
    task_id: str
    subagent: Literal["web_researcher", "document_analyst"]
    facets: list[Facet]
    status: Literal["success", "empty", "partial", "failed"]
    failure_type: FailureType | None = None
    retryable: bool | None = None
    attempted: list[str] = Field(default_factory=list)
    failed_queries: list[str] = Field(default_factory=list)
    empty_queries: list[str] = Field(default_factory=list)
    claims: list[Claim] = Field(default_factory=list)
    alternatives: list[str] = Field(default_factory=list)
    coverage_impact: str | None = None
    message: str | None = None

    @model_validator(mode="after")
    def consistent(self):
        failing = self.status in ("partial", "failed")
        if failing and (self.failure_type is None or self.retryable is None):
            raise ValueError("partial/failed results must say what failed and whether a retry can help")
        if not failing and self.failure_type is not None:
            raise ValueError("success/empty results must not carry a failure_type")
        if self.status == "partial" and not self.claims:
            raise ValueError("partial means some findings survived; use 'failed' otherwise")
        if self.status == "empty" and self.claims:
            raise ValueError("an empty result cannot contain claims")
        return self


class Brief(BaseModel):
    task_id: str
    subagent: Literal["web_researcher", "document_analyst"]
    facets: list[Facet]
    objective: str
    queries: list[str]
    documents: list[str]


# A small source registry: P1 is a press release, N1 a news article that only re-reports it.
REGISTRY = {s.source_id: s for s in [
    Source(source_id="P1", title="Shared mobility in 2024: the numbers", published="2025-02-10",
           methodology="validated trip records", derived_from=None),
    Source(source_id="N1", title="Scooter trips hit a record", published="2025-02-12", methodology=None, derived_from="P1"),
    Source(source_id="A1", title="Trip count audit 2024", published="2025-03-20",
           methodology="independent audit of trip records", derived_from=None),
    Source(source_id="O1", title="Riverton market update", published="2025-03-05",
           methodology="operator app data (every unlock counted)", derived_from=None),
    Source(source_id="U1", title="Our city, cleaner air", published="2025-04-15", methodology=None, derived_from=None),
    Source(source_id="L1", title="Lifecycle emissions working paper", published=None,
           methodology="lifecycle assessment", derived_from=None),
]}


def mk(cid: str, value: float | None, period: str | None, *sources: str, metric: str = "trips per year",
       unit: str = "million trips", subject: str = "shared micromobility trips", facet: str = "ridership") -> Claim:
    return Claim(claim_id=cid, facet=facet, statement=f"{metric} {value} {unit} in {period}", subject=subject,
                 metric=metric, value=value, unit=unit, period=period,
                 evidence=[Evidence(source_id=s, excerpt=f"{s} reports {value} {unit} for {period}.", location=None, verbatim=True)
                           for s in sources])


def origins(c: Claim) -> set[str]:
    return {(REGISTRY[e.source_id].derived_from if e.source_id in REGISTRY else None) or e.source_id for e in c.evidence}


def methods_of(c: Claim) -> set[str | None]:
    return {REGISTRY[o].methodology if o in REGISTRY else None for o in origins(c)}


def classify_pair_v1(a: Claim, b: Claim) -> str:
    """The practice notebook's classifier (Step 6)."""
    if (a.subject.lower(), (a.metric or "").lower(), (a.unit or "").lower()) != \
       (b.subject.lower(), (b.metric or "").lower(), (b.unit or "").lower()):
        return "different_quantity"
    if a.value is None or b.value is None:
        return "not_numeric"
    if a.period is None or b.period is None:
        return "undetermined"
    if a.period != b.period:
        return "temporal_difference"
    if abs(a.value - b.value) <= 0.05 * max(abs(a.value), abs(b.value)):
        return "agree"
    ma, mb = methods_of(a), methods_of(b)
    if None not in ma | mb and ma != mb:
        return "methodology_difference"
    return "conflict"


def failed_envelope(brief: Brief, failure_type: str, retryable: bool, message: str) -> SubagentResult:
    return SubagentResult(task_id=brief.task_id, subagent=brief.subagent, facets=brief.facets, status="failed",
                          failure_type=failure_type, retryable=retryable, attempted=brief.queries or brief.documents,
                          failed_queries=brief.queries or brief.documents, message=message,
                          coverage_impact=f"No findings for {', '.join(brief.facets)} from this task.")


def tag(text: str, name: str) -> str:
    m = re.search(rf"<{name}>(.*?)</{name}>", text, re.S)
    return m.group(1).strip() if m else ""


REF = re.compile(r"\[(C\d+|gap:[a-z]+)\]")
NUM = re.compile(r"(?<![A-Za-z\d.,])\d+(?:[.,]\d+)*")


def nums(text: str) -> set[float]:
    """Numbers in a text as floats; '1,900' -> 1900.0; ignores digits glued to letters such as CO2."""
    return {float(n.replace(",", "")) for n in NUM.findall(text)}


def split_sentences(text: str) -> list[str]:
    return [s for s in re.split(r"(?<=[.!?])\s+(?=[A-Z\d])", text.strip()) if s]


print("helpers ready")

## Part A — Quiz (10 questions)

Pick one letter per question. Scenario: the multi-agent research system (CCAR-F domains D1, D2 and D5), on both stacks.

**Q1. (Multi-agent research system)** After the first round, your coordinator re-delegates the economy facet because one search timed out. The retry subagent runs the same failing query again and returns findings that the first round already had. The logs show the retry prompt was only: "Retry the economy research." What is the most effective fix?

- A) Give subagents read access to the coordinator's full conversation history so they can see earlier rounds.
- B) Raise the retry subagent's turn limit so it has room to work out what failed before.
- C) Build the retry brief from the first envelope: the partial findings, the failed queries, and the alternative query the subagent suggested.
- D) Retry the identical brief up to three times with exponential backoff.

**Q2. (Multi-agent research system)** Traces show your coordinator calls the delegation tool once, waits for the result, then calls it again: four turns for four independent subagents, so the brief takes the sum of their latencies. Which change gives the largest latency gain without changing the results?

- A) Have the coordinator emit all four delegation calls in one response, run them concurrently, and return all four results in one message.
- B) Move every subagent to a smaller, faster model.
- C) Merge the four subagents into one subagent that researches every facet.
- D) Raise the coordinator's `max_tokens` so it can plan all four tasks in more detail.

**Q3. (Multi-agent research system)** The synthesizer was given the web researcher's search tool "in case it needs more context". Reviewers now find report sentences that cite sources no subagent retrieved and the coordinator never vetted. What is the best fix?

- A) Add "only search when absolutely necessary" to the synthesizer's prompt.
- B) Post-process the report to delete citations whose source IDs are not in the registry.
- C) Keep the tool, but log every search the synthesizer makes for later review.
- D) Remove the search tool from the synthesizer, pass it every claim record it needs in its prompt, and send any need for new research back through the coordinator.

**Q4. (Multi-agent research system)** A web subagent searched for "Riverton scooter fines 2024". The search ran successfully and matched nothing. The subagent returns status `failed`, failure type `unavailable`, `retryable: true`, and the coordinator retries the query three times. What is the correct handling?

- A) Keep `failed`, but set `retryable` to false so the coordinator stops retrying.
- B) Report the query as a valid empty result (status `empty`, or listed under empty queries in a successful envelope); the coordinator accepts it and the report says "searched, no sources found".
- C) Drop the query from the envelope entirely, because it produced nothing.
- D) Have the subagent keep retrying locally until something is found.

**Q5. (Multi-agent research system, OpenAI Agents SDK)** Your manager agent calls specialist agents through `Agent.as_tool()`. When a nested run fails on a model timeout, the manager receives only a generic "an error occurred while running the tool" message, and the report never mentions the gap. What should you change first?

- A) Replace agents-as-tools with handoffs, so the failing specialist can explain the problem to the user itself.
- B) Wrap the whole manager run in `try`/`except` and abort the brief whenever any specialist fails.
- C) Pass a `failure_error_function` to `as_tool()` that returns a structured envelope (failure type, the brief that was attempted, retryable, alternatives), so the manager and your code can decide on recovery and annotate coverage.
- D) Raise the specialist's `max_turns` so it gets more chances to finish.

**Q6. (Multi-agent research system)** The document analyst reads a hospital registry (38 injuries per million trips in 2024, counted from emergency admissions) and a council review (21 per million trips in 2024, counted from police reports). Its prompt says "report the most authoritative figure", so the report shows only 38. What should change?

- A) Average the two figures so the report shows one number.
- B) Report whichever source was published more recently.
- C) Leave both figures out until the sources agree.
- D) Report both values with their sources, publication dates, data periods and methods, label the pair as a methodology difference, and let the coordinator decide how the report presents it.

**Q7. (Multi-agent research system)** The report flags a "contradiction": 2.7 million trips versus 4.1 million trips, both from one press release published in February 2025. The claim records have no field for the period a number describes. What is the root-cause fix?

- A) Add a required, nullable data-period field to every claim record, and classify pairs by period before calling anything a conflict.
- B) Tell the synthesizer to resolve contradictions by keeping the larger value.
- C) Keep only the figure that appears first in the source.
- D) Use the publication date as the period for every claim from that source.

**Q8. (Multi-agent research system, OpenAI)** Your web researcher uses the hosted `web_search` tool. The extractor stores the text between each `url_citation`'s `start_index` and `end_index` as a "verbatim quote from the page". A reviewer finds a number in one "quote" that does not appear on the cited page. What is the correct understanding?

- A) `start_index` and `end_index` are character offsets into the fetched page, so the page must have changed since the search.
- B) A `url_citation` points at a URL and at a span of the model's own answer, not at page text. Store it as non-verbatim evidence, and check numbers against the fetched page before publishing.
- C) `url_citation` spans are always verbatim, so the reviewer must have read a different version of the page.
- D) Switch the web researcher to `file_citation` annotations, which include page quotes for web results.

**Q9. (Multi-agent research system, Claude)** To get claim records with quotes from a single call, a team sends the attached reports as `document` blocks with citations enabled and also sets `output_config.format` to a claim schema. Every request returns a 400. What is the best design?

- A) Retry the request with exponential backoff.
- B) Disable citations and ask the model to copy the quotes and source names into the JSON fields.
- C) Put the JSON schema in the system prompt and parse the cited text with regular expressions.
- D) Use two steps: a request with citations enabled produces cited text, code converts the citation blocks into evidence, and a separate structured-output request extracts the remaining fields from the excerpts that are already cited.

**Q10. (Evaluation)** Your eval reports 100% "sentence citation coverage": every summary sentence ends with at least one claim ID. A reviewer still finds a sentence stating 2,650 tonnes of CO2, while the cited claims say 1,900 and 3,400. Which addition to the eval would catch this kind of error?

- A) Count how many distinct sources the report cites.
- B) Ask the synthesizer to rate its own confidence for each sentence, and flag scores below 0.8.
- C) A citation-precision metric that checks, for each sentence and cited claim, that the evidence really is in the source and that every number in the sentence appears in the cited evidence.
- D) Facet coverage: the share of required facets that have at least one claim.

In [ ]:
answers = {
    "q1": None,
    "q2": None,
    "q3": None,
    "q4": None,
    "q5": None,
    "q6": None,
    "q7": None,
    "q8": None,
    "q9": None,
    "q10": None,
}

In [ ]:
ANSWER_HASHES = {'q1': '227effd0f0211c628c0ae190f31feffeed3e6aa4b570e9bd60b91ac5bb2bb16e', 'q2': '3d9c97ee45947882f5eee5cf878c4f33a0a270d8f2a2e1fc259d66cdc95c7b45', 'q3': '14099ffd7333e442b90669f0e435e505e0e45a0ed148540caca94158c15a27c6', 'q4': '61f643a360ded8f53a0309d76dab6c7178f5a30db61b94cdcd57e7341a5aeaaa', 'q5': 'e3c6d5ddd445920945724cd165c914deaf97d5041f947ba2df35e569269b353f', 'q6': '70425e0bfb4a573a030df7909d2bae441e3fdb1e7a565e097f3118231f4f27a9', 'q7': '8722338db4eb12eaae5f6f26c68875af2b115581d770f01f98b9e0e1e6010ac1', 'q8': '39f95f59afaa9522616af26bd34df1e0184030ae8f738e319fc9a1f15b075834', 'q9': 'e58517837d895fe6365a5801be9a66d9d1d39f1d863920a84cbed549cfec1954', 'q10': 'd6ea6e76ddfa07a3b78a8bd000db76a223a8ccc3f38e290cf5a68d95e5494cc5'}


def grade_quiz(answers: dict) -> int:
    correct = 0
    for qid in sorted(ANSWER_HASHES, key=lambda q: int(q[1:])):
        given = answers.get(qid)
        letter = given.strip().upper() if isinstance(given, str) else None
        ok = letter is not None and hashlib.sha256(f"{qid}:{letter}".encode()).hexdigest() == ANSWER_HASHES[qid]
        correct += ok
        print(f"{'✓' if ok else '✗'} {qid}: {letter or '(no answer)'}")
    SCORE["quiz"] = correct
    print(f"Quiz: {correct}/{len(ANSWER_HASHES)} = {correct / len(ANSWER_HASHES):.0%}")
    return correct


_ = grade_quiz(answers)

## Part B — Auto-checked extensions (offline)

Each exercise: read the task, replace the stub, run the checker. Exercises 4 and 5 are the two extensions the capstone asks for: a new **source-conflict rule** and **coverage annotations** for a report.

### Exercise 1 — A brief that carries all the context (R2)

Write `render_brief(brief, prior=(), failed=())` returning the subagent's whole prompt. The subagent sees nothing else, so it must contain:

- the header `<task id="{task_id}" subagent="{subagent}">` and a closing `</task>`;
- `<goal>` with `QUESTION`, `<as_of>` with `AS_OF`, `<your_facets>` listing the brief's facets, `<objective>`;
- `<out_of_scope>` listing **every** required facet that is **not** the brief's (and none of its own);
- `<suggested_queries>` and `<documents>` with the brief's lists;
- `<prior_findings>`: the text `none` when there are no prior claims; otherwise, for every prior claim, its statement and for each evidence item the `source_id`, the excerpt **verbatim**, and the source's **publication date** from `REGISTRY` (the subagent cannot look dates up);
- `<already_failed>` with the failed queries; `<output_contract>` with `CONTRACT`.

Any format inside the tags is fine (JSON or lines). Theory: [R2](README.md#r2-explicit-context-passing)

In [ ]:
def render_brief(brief: Brief, prior: list[Claim] = (), failed: list[str] = ()) -> str:
    # TODO: build the self-contained brief described above
    raise NotImplementedError

In [ ]:
def check_ex1() -> None:
    def check():
        b = Brief(task_id="T2-R1", subagent="web_researcher", facets=["economy"], objective="Find local jobs and revenue.",
                  queries=["Riverton micromobility permit fees"], documents=[])
        first = render_brief(b)
        assert isinstance(first, str), "render_brief must return a string"
        assert first.strip().startswith('<task id="T2-R1" subagent="web_researcher">') and first.strip().endswith("</task>"), \
            'start with <task id="..." subagent="..."> and end with </task>'
        assert tag(first, "goal") == QUESTION, "<goal> must hold QUESTION exactly: the subagent has no other way to know the goal"
        assert tag(first, "as_of") == AS_OF, "<as_of> must hold AS_OF (the model's own sense of 'today' is its training cutoff)"
        assert "economy" in tag(first, "your_facets"), "<your_facets> must list the brief's facets"
        oos = tag(first, "out_of_scope")
        missing = [f for f in REQUIRED_FACETS if f != "economy" and f not in oos]
        assert not missing, f"<out_of_scope> must list every facet the brief does NOT own; missing {missing}"
        assert "economy" not in oos, "<out_of_scope> must not contain the brief's own facets"
        assert "Riverton micromobility permit fees" in tag(first, "suggested_queries"), "<suggested_queries> must hold the queries"
        assert tag(first, "prior_findings") == "none", "with no prior claims, <prior_findings> must be exactly 'none'"
        assert tag(first, "output_contract"), "<output_contract> must not be empty"
        prior = [mk("C7", 140, "2024", "P1", metric="local staff", unit="jobs", facet="economy"),
                 mk("C8", 4.1, "2024", "N1")]
        text = render_brief(b, prior, ["Riverton micromobility tax revenue"])
        pf = tag(text, "prior_findings")
        for c in prior:
            assert c.statement in pf, f"<prior_findings> must include each prior claim's statement ({c.claim_id} missing)"
            for e in c.evidence:
                assert e.excerpt in pf, f"<prior_findings> must include the excerpt verbatim ({c.claim_id})"
                assert e.source_id in pf, f"<prior_findings> must name the source_id ({c.claim_id})"
                assert REGISTRY[e.source_id].published in pf, \
                    f"<prior_findings> must include {e.source_id}'s publication date from REGISTRY: the subagent cannot look it up"
        assert "Riverton micromobility tax revenue" in tag(text, "already_failed"), \
            "<already_failed> must list the failed queries, or the retry will run them again"
    _run_check(1, check)


check_ex1()

### Exercise 2 — Bounded fan-out that survives failures (R3, R5)

Write `async def fan_out(briefs, worker, limit=3)` that runs `await worker(brief)` for every brief **concurrently**, with **at most `limit` running at once**, and returns the `SubagentResult`s **in the same order as `briefs`**. A worker that raises must not cancel or lose the others; turn its exception into an envelope with the provided `failed_envelope(brief, failure_type, retryable, message)`:

| Exception | `failure_type` | `retryable` |
|---|---|---|
| `TimeoutError` | `"timeout"` | `True` |
| `PermissionError` | `"permission_denied"` | `False` |
| anything else | `"internal_error"` | `False` |

The checker is a coroutine: run it with `await check_ex2()`. Theory: [R3](README.md#r3-parallel-fan-out), [R5](README.md#r5-structured-error-propagation)

In [ ]:
async def fan_out(briefs: list[Brief], worker, limit: int = 3) -> list[SubagentResult]:
    # TODO: asyncio.Semaphore + asyncio.gather; catch per worker; keep the order
    raise NotImplementedError

In [ ]:
async def check_ex2() -> None:
    async def check():
        state = {"now": 0, "max": 0, "done": []}

        async def worker(b: Brief) -> SubagentResult:
            state["now"] += 1
            state["max"] = max(state["max"], state["now"])
            try:
                await asyncio.sleep(0.12)
                if b.task_id == "T2":
                    raise TimeoutError("search backend timed out")
                if b.task_id == "T5":
                    raise PermissionError("403 from the licensed archive")
                if b.task_id == "T6":
                    raise ValueError("model returned invalid JSON")
                await asyncio.sleep(0.02)
                state["done"].append(b.task_id)
                return SubagentResult(task_id=b.task_id, subagent=b.subagent, facets=b.facets, status="empty", attempted=b.queries)
            finally:
                state["now"] -= 1

        briefs = [Brief(task_id=f"T{i}", subagent="web_researcher", facets=["economy"], objective="o",
                        queries=[f"query {i}"], documents=[]) for i in range(1, 8)]
        t0 = time.perf_counter()
        pending = fan_out(briefs, worker, limit=3)
        assert inspect.isawaitable(pending), "fan_out must be `async def`"
        out = await pending
        elapsed = time.perf_counter() - t0
        assert isinstance(out, list) and len(out) == 7, f"expected 7 results, got {out!r:.80}"
        assert [r.task_id for r in out] == [b.task_id for b in briefs], "results must come back in the order of the briefs"
        assert state["max"] <= 3, f"{state['max']} workers ran at once with limit=3: bound concurrency with asyncio.Semaphore(limit)"
        assert state["max"] == 3 and elapsed < 0.8, \
            f"at most {state['max']} worker(s) ran at once ({elapsed:.2f}s): run them concurrently (asyncio.gather), not one by one"
        assert sorted(state["done"]) == ["T1", "T3", "T4", "T7"], \
            "a failing worker cancelled healthy ones: catch per worker and use gather, not a TaskGroup"
        by = {r.task_id: r for r in out}
        assert (by["T2"].status, by["T2"].failure_type, by["T2"].retryable) == ("failed", "timeout", True), \
            "TimeoutError -> failed envelope, failure_type 'timeout', retryable True"
        assert (by["T5"].failure_type, by["T5"].retryable) == ("permission_denied", False), \
            "PermissionError -> 'permission_denied', retryable False (a retry cannot grant access)"
        assert (by["T6"].failure_type, by["T6"].retryable) == ("internal_error", False), "other errors -> 'internal_error', retryable False"
        assert by["T2"].attempted == ["query 2"], "use failed_envelope so the envelope says what was attempted"
    await _run_check_async(2, check)


await check_ex2()

### Exercise 3 — Compute the envelope from the tool log (R5)

Write `envelope_from_log(brief, claims, log)` where `log` is a list of `(query, outcome)` and `outcome` is one of `"hits"`, `"empty"`, `"timeout"`, `"unavailable"`, `"permission_denied"`. Status comes from the log, not from the model:

- `attempted`: every query in order; `failed_queries`: queries whose outcome is not `hits`/`empty`; `empty_queries`: queries with outcome `empty` (zero matches is a **finding**, not a failure).
- No failures: `status="success"` if there are claims, else `"empty"`.
- Failures: `status="partial"` if there are claims, else `"failed"`; `failure_type` = the outcome of the **first** failed query; `retryable=True` only for `timeout` and `unavailable`; `coverage_impact` names every failed query.

Theory: [R5](README.md#r5-structured-error-propagation)

In [ ]:
def envelope_from_log(brief: Brief, claims: list[Claim], log: list[tuple[str, str]]) -> SubagentResult:
    # TODO
    raise NotImplementedError

In [ ]:
def check_ex3() -> None:
    def check():
        b = Brief(task_id="T3", subagent="web_researcher", facets=["regulation"], objective="o", queries=[], documents=[])
        c = [mk("C1", 20, "2024-12", "P1", metric="speed limit", unit="km/h", facet="regulation")]

        def envelope_from_log_checked(claims, log, case):
            try:
                return envelope_from_log(b, claims, log)
            except ValidationError as exc:
                raise AssertionError(f"case '{case}': SubagentResult rejected your envelope ({exc.errors()[0]['msg']}). "
                                     "Only outcomes other than hits/empty are failures, and a failing envelope needs "
                                     "status partial/failed, a failure_type and retryable") from None

        r = envelope_from_log_checked(c, [("speed cap", "hits"), ("fines", "empty")], "one hit, one empty query")
        assert isinstance(r, SubagentResult), "return a SubagentResult"
        assert r.status == "success" and r.empty_queries == ["fines"] and r.failed_queries == [], \
            "a query with zero matches is a valid empty result: success overall, listed in empty_queries, not failed_queries"
        assert r.attempted == ["speed cap", "fines"], "attempted lists every query in order"
        r = envelope_from_log_checked([], [("fines", "empty")], "only an empty query")
        assert r.status == "empty" and r.failure_type is None, "no claims and no failures -> status 'empty'"
        r = envelope_from_log_checked(c, [("speed cap", "hits"), ("tax revenue", "timeout")], "claims plus a timeout")
        assert r.status == "partial" and r.failure_type == "timeout" and r.retryable is True, \
            "claims plus a failure -> 'partial' with the failure type and retryable=True for a timeout"
        assert r.failed_queries == ["tax revenue"] and "tax revenue" in (r.coverage_impact or ""), \
            "failed_queries and coverage_impact must name the query that failed"
        r = envelope_from_log_checked([], [("archive", "permission_denied"), ("tax revenue", "timeout")], "two failures, no claims")
        assert r.status == "failed" and r.failure_type == "permission_denied" and r.retryable is False, \
            "first failure decides failure_type; permission_denied is not retryable"
        r = envelope_from_log_checked([], [("fines", "empty"), ("registry", "unavailable")], "an empty query and a failed query")
        assert r.status == "failed" and r.empty_queries == ["fines"] and r.retryable is True, \
            "an empty query plus a failed query and no claims is 'failed' (not 'empty'); 'unavailable' is retryable"
    _run_check(3, check)


check_ex3()

### Exercise 4 — Add a source-conflict rule (R7)

The practice notebook's classifier, `classify_pair_v1(a, b)`, compares methods by independent origin. It still has a blind spot. A news article (`N1`, `derived_from="P1"`) misquotes its origin: it says 4.6 million trips where the press release says 4.1 million, for the same year. `v1` calls that a `"conflict"` between two sources, but it is one source copied wrongly.

1. Write `classify_pair_v2(a, b)` that returns `classify_pair_v1(a, b)`, **except** that when `v1` returns `"conflict"` or `"methodology_difference"` **and both claims trace to the same independent origins** (`origins(a) == origins(b)`), it returns `"transcription_error"`. Keep `v1`'s order: different periods are still a trend, values within 5% still agree.
2. Write `conflict_record(a, b, kind)` returning a dict with `"kind"`, `"claims"` (`[a.claim_id, b.claim_id]`), `"sides"` (one dict per claim with `"claim_id"`, `"value"`, `"unit"`, `"period"`, `"sources"` = the claim's own evidence source IDs, `"published"` = their publication dates from `REGISTRY` with `"date unknown"` when `None`, and `"methods"` with `"method not stated"` when `None`), and `"resolution"`:
   - `transcription_error`: `"trust <origin>; flag <other source IDs> for correction"`
   - `conflict`: a string starting with `"unresolved"`
   - `methodology_difference`: a string starting with `"explained"`

Theory: [R7](README.md#r7-conflict-handling-with-both-sources-and-dates)

In [ ]:
def classify_pair_v2(a: Claim, b: Claim) -> str:
    # TODO: start from classify_pair_v1(a, b) and add the transcription-error rule
    raise NotImplementedError


def conflict_record(a: Claim, b: Claim, kind: str) -> dict:
    # TODO
    raise NotImplementedError

In [ ]:
def check_ex4() -> None:
    def check():
        p = mk("C1", 4.1, "2024", "P1")
        n_bad = mk("C2", 4.6, "2024", "N1")
        assert classify_pair_v2(p, n_bad) == "transcription_error", \
            "N1 only re-reports P1: same independent origin, different value -> 'transcription_error'"
        assert classify_pair_v2(p, mk("C3", 5.6, "2024", "O1")) == "methodology_difference", \
            "independent sources with known, different methods stay 'methodology_difference'"
        assert classify_pair_v2(p, mk("C4", 6.0, "2024", "U1")) == "conflict", \
            "independent sources, unknown method -> still 'conflict'"
        assert classify_pair_v2(mk("C5", 2.7, "2023", "P1"), n_bad) == "temporal_difference", \
            "different periods are a trend even when one source derives from the other: call classify_pair_v1 first"
        assert classify_pair_v2(p, mk("C6", 4.1, "2024", "N1")) == "agree", "equal values from origin and derivative agree"
        rec = conflict_record(p, n_bad, "transcription_error")
        assert rec.get("kind") == "transcription_error" and rec.get("claims") == ["C1", "C2"], "record needs kind and claims"
        assert "P1" in rec.get("resolution", "") and "N1" in rec["resolution"] and rec["resolution"].startswith("trust P1"), \
            "transcription_error resolution: 'trust <origin>; flag <derivative> for correction'"
        s0, s1 = rec["sides"]
        assert (s0["value"], s1["value"]) == (4.1, 4.6) and s0["sources"] == ["P1"] and s1["sources"] == ["N1"], \
            "each side keeps its own value and its own evidence sources"
        assert s0["published"] == ["2025-02-10"] and s1["published"] == ["2025-02-12"], "each side needs its sources' publication dates"
        assert s1["methods"] == ["method not stated"], "a missing method must read 'method not stated', never be dropped"
        rec2 = conflict_record(mk("C7", 1900, "2024", "L1", metric="CO2 avoided", unit="tonnes", subject="co2", facet="emissions"),
                               mk("C8", 3400, "2024", "U1", metric="CO2 avoided", unit="tonnes", subject="co2", facet="emissions"),
                               "conflict")
        assert rec2["sides"][0]["published"] == ["date unknown"], "a missing publication date must read 'date unknown', never be guessed"
        assert rec2["resolution"].startswith("unresolved"), "a genuine conflict stays unresolved for the coordinator"
        assert conflict_record(p, mk("C3", 5.6, "2024", "O1"), "methodology_difference")["resolution"].startswith("explained")
    _run_check(4, check)


check_ex4()

### Exercise 5 — Add coverage annotations to a report (R6)

A teammate's renderer produced a report with one `## <Facet>` section per facet it found (`## Ridership`, `## Economy`, …) and no coverage information. Write `annotate_coverage(report, plan_facets, results)` that returns a **new** report string:

1. For each planned facet, compute a label from the envelopes whose `facets` include it:
   - no envelope covers it → `NO COVERAGE`, reason `not searched`;
   - any `failed_queries` → `PARTIAL COVERAGE` if some claim in those envelopes has that facet, else `NO COVERAGE`; reason `"<failure_type> on 'q1', 'q2'"` (the failure type of the first failing envelope, every failed query in single quotes, joined by `", "`);
   - otherwise `FULL COVERAGE` with no reason (even if some queries were empty).
2. Rewrite each matching heading (case-insensitive facet name) to `## Economy (PARTIAL COVERAGE: timeout on 'q')` or `## Ridership (FULL COVERAGE)`. A planned facet with no section gets a new section `## Safety (NO COVERAGE: not searched)` appended.
3. Append `## Coverage and limitations` with one line per planned facet: `- economy: PARTIAL COVERAGE (timeout on 'q')`; when the facet has empty queries, add `; searched, no sources found: 'q1', 'q2'`.
4. Keep every other line of the report unchanged. Never write "no evidence".

Theory: [R6](README.md#r6-coverage-annotations)

In [ ]:
def annotate_coverage(report: str, plan_facets: list[str], results: list[SubagentResult]) -> str:
    # TODO
    raise NotImplementedError

In [ ]:
def check_ex5() -> None:
    def check():
        report = ("# Riverton brief\n\n## Summary\n\nTrips grew [C1].\n\n## Ridership\n\n- 4.1 million trips [C1]\n\n"
                  "## Economy\n\n- 140 local staff [C2]\n\n## Regulation\n\n- 20 km/h [C3]\n")
        jobs = mk("C2", 140, "2024", "P1", metric="local staff", unit="jobs", facet="economy")
        speed = mk("C3", 20, "2024-12", "P1", metric="speed limit", unit="km/h", facet="regulation")
        results = [
            SubagentResult(task_id="T1", subagent="web_researcher", facets=["ridership"], status="success",
                           attempted=["trips"], claims=[mk("C1", 4.1, "2024", "P1")]),
            SubagentResult(task_id="T2", subagent="web_researcher", facets=["economy"], status="partial",
                           failure_type="timeout", retryable=True, attempted=["jobs", "tax revenue"],
                           failed_queries=["tax revenue"], claims=[jobs]),
            SubagentResult(task_id="T3", subagent="web_researcher", facets=["regulation"], status="success",
                           attempted=["speed cap", "fines 2024"], empty_queries=["fines 2024"], claims=[speed]),
        ]
        before = [r.model_dump() for r in results]
        out = annotate_coverage(report, ["ridership", "economy", "regulation", "safety"], results)
        assert isinstance(out, str), "return the new report as a string"
        assert not re.search(r"no evidence", out, re.I), "never write 'no evidence': a failed or unsearched source is a gap, not a finding"
        lines = out.split("\n")
        assert "## Ridership (FULL COVERAGE)" in lines, "a facet whose searches all ran is '## Ridership (FULL COVERAGE)'"
        assert "## Economy (PARTIAL COVERAGE: timeout on 'tax revenue')" in lines, \
            "economy has a claim and a failed query: '## Economy (PARTIAL COVERAGE: timeout on 'tax revenue')'"
        assert "## Regulation (FULL COVERAGE)" in lines, \
            "an EMPTY query is a finding, not a failure: regulation stays FULL COVERAGE"
        assert "## Safety (NO COVERAGE: not searched)" in lines, "a planned facet nobody covered gets '## Safety (NO COVERAGE: not searched)'"
        assert "## Summary" in lines and "- 140 local staff [C2]" in lines and "Trips grew [C1]." in lines, \
            "keep every other line of the report unchanged"
        cov = out.split("## Coverage and limitations", 1)
        assert len(cov) == 2, "append a '## Coverage and limitations' section"
        cov = cov[1]
        assert "- economy: PARTIAL COVERAGE (timeout on 'tax revenue')" in cov, "coverage line for economy is missing or differs"
        assert "- regulation: FULL COVERAGE; searched, no sources found: 'fines 2024'" in cov, \
            "the regulation line must say the empty query was 'searched, no sources found'"
        assert "- safety: NO COVERAGE (not searched)" in cov and "- ridership: FULL COVERAGE" in cov
        assert [r.model_dump() for r in results] == before, "do not mutate the envelopes"
        failed_only = [results[0], SubagentResult(task_id="T2", subagent="web_researcher", facets=["economy"], status="failed",
                                                  failure_type="unavailable", retryable=True, attempted=["a", "b"],
                                                  failed_queries=["a", "b"])]
        out2 = annotate_coverage(report, ["ridership", "economy"], failed_only)
        assert "## Economy (NO COVERAGE: unavailable on 'a', 'b')" in out2.split("\n"), \
            "failures and no surviving claims -> NO COVERAGE, with every failed query listed"
    _run_check(5, check)


check_ex5()

### Exercise 6 — Citation precision (evaluation)

Write `citation_precision(summary, claims, source_texts)` from the [evaluation plan](README.md#evaluation-plan). Split the summary with `split_sentences`; every `[C#]` reference in a sentence is one **(sentence, claim) pair**. A pair is supported when **all** of these hold:

1. the claim ID exists in `claims`;
2. the claim's evidence checks out against `source_texts[source_id]`: a `verbatim=True` excerpt must be a substring of the source text; for `verbatim=False` the claim's `value` must be among the source text's numbers (`nums(text)`); an unknown source fails;
3. every number in the sentence (use `nums(REF.sub("", sentence))`) appears in the **cited evidence** of the claims cited in that sentence (any of them), or in their `period`s. The cited evidence of a `verbatim=True` item is its excerpt; for a `verbatim=False` item (an answer span plus a source) it is the sentences of the source text (`split_sentences`) that contain the claim's `value`. Numbers found elsewhere in the source document do **not** count: a page that mentions 140 staff does not support "140 million trips".

Return supported pairs ÷ all pairs, or `0.0` when there are no pairs. Sentences without references add no pairs (sentence citation coverage is a separate metric).

In [ ]:
def citation_precision(summary: str, claims: list[Claim], source_texts: dict[str, str]) -> float:
    # TODO
    raise NotImplementedError

In [ ]:
def check_ex6() -> None:
    import math

    def check():
        texts = {"S1": "Riders took 4.1 million trips in 2024.", "S2": "Operators recorded 5.6 million rides in 2024.",
                 "S3": "The fleet avoided 1,900 tonnes of CO2 in 2024."}

        def claim(cid, value, sid, excerpt, verbatim=True):
            return Claim(claim_id=cid, facet="ridership", statement=excerpt, subject="s", metric="m", value=value, unit="u",
                         period="2024", evidence=[Evidence(source_id=sid, excerpt=excerpt, location=None, verbatim=verbatim)])

        claims = [claim("C1", 4.1, "S1", "Riders took 4.1 million trips in 2024."),
                  claim("C2", 5.6, "S2", "Operators recorded 5.6 million rides", verbatim=False),
                  claim("C3", 1900, "S3", "avoided 1,900 tonnes"),
                  claim("C4", 1900, "S3", "avoided 1,900 tonnes in total"),
                  claim("C5", 7.7, "S2", "Another count gives 7.7 million", verbatim=False)]
        summary = ("Riders took 4.1 million trips in 2024 [C1]. "
                   "Counts differ: 4.1 million [C1] versus 5.6 million rides [C2]. "
                   "The fleet avoided 1,950 tonnes of CO2 in 2024 [C3]. "
                   "Emissions savings reached 1,900 tonnes [C4]. "
                   "Another count gives 7.7 million [C5]. "
                   "Ridership doubled [C9]. "
                   "This sentence cites nothing.")
        got = citation_precision(summary, claims, texts)
        assert isinstance(got, float) and math.isfinite(got) and 0.0 <= got <= 1.0, \
            f"return a finite float between 0 and 1, got {got!r}"
        expected = 3 / 7
        if abs(got - expected) > 1e-9:
            hints = [(4 / 7, "a pair is unsupported if the sentence has a number missing from the evidence (1,950 vs 1,900), "
                              "or if the cited excerpt is not really in the source (C4)"),
                     (1 / 7, "numbers may come from ANY claim cited in the same sentence (4.1 from C1, 5.6 from C2)"),
                     (2 / 6, "count (sentence, claim) PAIRS, not sentences"),
                     (3 / 8, "a sentence without references adds no pairs"),
                     (5 / 7, "a verbatim excerpt that is not in the source (C4) or a non-verbatim value missing from it (C5) is unsupported")]
            hint = next((h for v, h in hints if abs(got - v) < 1e-9), "recheck the three rules")
            raise AssertionError(f"expected 3/7 = {expected:.3f}, got {got:.3f}: {hint}")
        assert citation_precision("", claims, texts) == 0.0, "no pairs -> 0.0"
        assert citation_precision("Ridership doubled [C9].", claims, texts) == 0.0, "an unknown claim ID is unsupported"
        page = {"S4": "Riders took 4.1 million trips in 2024. The two operators employ about 140 local staff."}
        for c in (claim("C6", 4.1, "S4", "Riders took 4.1 million trips in 2024."),
                  claim("C6", 4.1, "S4", "Riders took 4.1 million trips", verbatim=False)):
            got = citation_precision("Riders took 140 million trips in 2024 [C6].", [c], page)
            assert got == 0.0, (f"'140 million trips' citing a 4.1-million-trips claim scored {got!r}: numbers must be in the "
                                "CITED evidence (the excerpt, or for verbatim=False the source sentence that states the claim's "
                                "value), not anywhere in the source document")
    _run_check(6, check)


check_ex6()

## Part C — Rubric self-scoring

Score your build on the [capstone rubric](README.md#rubric), 0–3 per dimension, and write the **evidence** next to each score (an eval number, a check result, a recorded prompt, a timing, a paragraph of your comparison). A dimension scores 2 or more only when the evidence covers **both** builds. Then answer the red-flag questions honestly: any `True` caps the result.

| Score | Meaning |
|---|---|
| 0 | Missing, or contradicts the brief |
| 1 | Present but not backed by evidence, or works only on the happy path |
| 2 | Meets the brief, backed by evidence from the checks or evals, in both stacks |
| 3 | Production-grade: also handles edge cases, explains trade-offs, would survive a senior architect's review |

In [ ]:
RUBRIC = {   # dimension: [weight, your score 0-3 (None = not scored yet), evidence]
    "Orchestration and context passing (D1)": [25, None, ""],
    "Tool and agent scoping (D2)":             [15, None, ""],
    "Error propagation and coverage (D5)":     [20, None, ""],
    "Provenance and conflicts (D5)":           [20, None, ""],
    "Evaluation":                              [10, None, ""],
    "Cost, latency and operations":            [5,  None, ""],
    "Communication":                           [5,  None, ""],
}
RED_FLAGS = {   # set each to True or False
    "A contested figure is stated as settled somewhere in the report": None,
    "A failed source is described as 'no evidence' or omitted": None,
    "A report sentence contains a number that is not in its evidence": None,
    "Subagents pass results to each other around the coordinator": None,
    "There are no eval numbers for one of the builds": None,
}

In [ ]:
def score_rubric() -> None:
    SCORE.pop("rubric", None)                    # never keep a result from an earlier, different scoring
    SCORE.pop("red_flags", None)
    missing = [d for d, (_, s, _) in RUBRIC.items() if s is None]
    unanswered = [f for f, v in RED_FLAGS.items() if v is None]
    if missing or unanswered:
        print("Not scored yet:", *(missing + unanswered), sep="\n  - ")
        return
    bad = [d for d, (_, s, ev) in RUBRIC.items()
           if isinstance(s, bool) or s not in (0, 1, 2, 3) or not isinstance(ev, str)]
    bad_flags = [f for f, v in RED_FLAGS.items() if not isinstance(v, bool)]
    if bad or bad_flags:
        print("Scores must be the integers 0, 1, 2 or 3 with text evidence; red flags must be True or False:",
              *(bad + bad_flags), sep="\n  - ")
        return
    no_evidence = [d for d, (_, s, ev) in RUBRIC.items() if s >= 2 and not ev.strip()]
    capped = {d: min(s, 1) if d in no_evidence else s for d, (_, s, _) in RUBRIC.items()}
    flags = [f for f, v in RED_FLAGS.items() if v]
    total = sum(RUBRIC[d][0] * s for d, s in capped.items())
    maximum = 3 * sum(w for w, _, _ in RUBRIC.values())
    SCORE["rubric"] = total / maximum
    SCORE["red_flags"] = len(flags)
    for d, s in capped.items():
        print(f"{d:42} {s}/3  x{RUBRIC[d][0]:>2}" + ("   (capped at 1: a 2 or 3 needs written evidence)" if d in no_evidence else ""))
    print(f"Weighted: {total}/{maximum} = {SCORE['rubric']:.0%}" + (f" | RED FLAGS: {flags}" if flags else ""))
    print("Rerun the Scorecard cell to update the final verdict.")


score_rubric()

## Part D — Design reviews

Two "what would you change if…" scenarios for the Riverton system. Write 10–15 lines for each, then open the model answer.

**Review 1: ten times the work, half the budget.** The policy unit is so pleased that the regional government wants the same brief for **12 cities**, delivered **every Monday by 08:00**, on **half** the per-brief budget. Analysts no longer need answers in two minutes. Several city archives rate-limit heavily, and two cities publish most of their reports as scanned PDFs. What do you change, on each stack, and what do you keep?

*Your answer here.*

<details><summary>Model answer: Review 1</summary>

- **Change the execution mode, not the architecture.** The work is no longer interactive, so move the fan-out to batch processing: Claude's Message Batches API (citations work with batches) or OpenAI's Batch API, at about half the price of synchronous calls. A batch cannot chain steps, so each stage (worker searches, field extraction, synthesis) becomes its own batch, keyed by `custom_id` = city + task ID, with resubmission of failed items. Keep a synchronous path for a re-run of one city.
- **Cost levers:** a smaller model for workers and field extraction, the stronger model only for coordination and synthesis; prompt caching for the shared prefix (system prompt, contract, the city-independent part of the brief); compact claim records between agents (no page dumps); the same plan template per city with city-specific queries, so the coordinator call is cheap or replaced by code.
- **Rate limits:** a per-source concurrency cap (semaphore per archive) and per-model limits; the SDKs' backoff on 429 using `retry-after`; a spend cap per run. On OpenAI, `max_function_tool_concurrency`; on the Agent SDK variant, `max_budget_usd` and spawn depth 1.
- **Scanned PDFs:** citations need extractable text. Image-only scans cannot be cited, so add an OCR step that produces text documents (and keep the original page number in `context`/attributes), or route those reports to a human. Coverage must say "scanned report not machine-readable" rather than silently skipping it.
- **Keep:** the contracts, envelopes, conflict classifier, coverage table, renderer, audit and eval suite unchanged. Add a **per-city segment** to every metric and a weekly trend of citation precision and gap counts. Store each week's claim store and reports so week-over-week changes are traceable.
- **Stack note:** OpenAI's hosted `file_search` with one vector store per city (attributes for city and publication date, filters per query) reduces retrieval code; on Claude you run your own retrieval and pass `search_result` blocks. Either way, the verbatim-evidence difference from the comparison table still applies.
</details>

**Review 2: the auditors arrive.** After a contested council session, the city's legal team issues a rule: every figure in a brief must trace to a **primary** source, contested figures need a **named human approver** before publication, and the city must be able to reproduce any brief for **five years**. One analyst also asks for a "confidence score" next to every finding. What do you change?

*Your answer here.*

<details><summary>Model answer: Review 2</summary>

- **Primary sources:** add `source_type` and lineage to the registry as hard data, and resolve every derivative (`derived_from`) to its origin. A figure whose only evidence is a derivative or advocacy source is either sent back for a primary source (a scoped `verify_fact` or re-delegation through the coordinator) or shown as "secondary only" and blocked from key figures. Use a source allow-list (domains, publishers) in the search tool and `allowed_domains` filters where the platform offers them.
- **Human approval:** contested records (`conflict`, `transcription_error`) and high-impact single-source figures go to a review queue *before* rendering; the approver's name, time, decision and note are stored in the conflict record's `reconciliation`. The audit refuses to publish while any contested record lacks an approval. Approval is enforced in code, not in a prompt.
- **Reproducibility for five years:** store the plan, every brief, every envelope, the claim store, the source registry with retrieval timestamps and content hashes (or archived copies), the raw cited responses, model IDs and settings, the synthesizer prompt and output, the audit result and the approvals. Pin model versions and record SDK versions; plan for model deprecation by storing outputs, not by expecting identical regeneration.
- **"Confidence scores":** do not show the model's self-rated confidence as if it were calibrated. Show **support levels derived from evidence** (well established = two independent primary origins agree; single source; contested; gap) and, if a numeric score is required, calibrate it against labeled reviews first. Explain the difference to the analyst.
- **Checks to add:** primary-source rate per brief, approval completeness, and an integrity re-check of stored excerpts against archived sources, all in the eval suite.
</details>

## Scorecard

In [ ]:
quiz = SCORE.get("quiz", 0)
n_ex = 6
passed = sum(RESULTS.get(i, False) for i in range(1, n_ex + 1))
overall = (quiz + passed) / (10 + n_ex)
print(f"Quiz:       {quiz}/10 ({quiz / 10:.0%})")
print(f"Exercises:  {passed}/{n_ex} passed  " + " ".join(f"{i}:{'✓' if RESULTS.get(i) else '✗'}" for i in range(1, n_ex + 1)))
print(f"Overall:    {overall:.0%}  (pass mark 72%)")
if "rubric" in SCORE:
    print(f"Rubric:     {SCORE['rubric']:.0%}  (pass mark 72%, no red flag)" + (f" | red flags: {SCORE['red_flags']}" if SCORE.get("red_flags") else ""))
else:
    print("Rubric:     not scored yet (Part C)")
ok = overall >= 0.72 and SCORE.get("rubric", 0) >= 0.72 and not SCORE.get("red_flags")
print("PASS: capstone complete" if ok else "RETRY: revisit the requirements behind your ✗ items and low rubric dimensions, then rerun")

Next capstone: [Capstone 03 — Structured extraction pipeline at scale](../03-extraction-pipeline-at-scale/README.md). Back to the [capstones overview](../README.md).

---

## Solutions (spoilers)

Open these only after you have attempted everything.

<details><summary>Quiz answers with explanations</summary>

**Q1:** **C.** Subagents know only what is in their prompt. A retry brief must carry the partial findings, the queries that already failed, and the specialist's alternative, so the retry does new work. A is not how subagent context works and would flood the subagent with irrelevant history. B and D repeat the same blind attempt.

**Q2:** **A.** Independent subagents spawned in one response run concurrently, so latency becomes roughly the slowest subagent instead of the sum. B trades quality for a smaller gain. C removes the parallelism and the context isolation. D does nothing for latency.

**Q3:** **D.** Scoped tools: an agent with tools outside its role tends to misuse them, and the synthesizer's job is to write from vetted records. New research goes back through the coordinator, which keeps control and coverage tracking. A is a probabilistic instruction. B hides the symptom after the damage. C observes the problem without preventing it.

**Q4:** **B.** A successful search with zero matches is a valid result and a finding ("searched, no sources found"). Reporting it as a failure wastes retries and misstates coverage. A keeps the wrong status. C loses information the report needs. D turns a finding into an endless loop.

**Q5:** **C.** The default tool error hides the failure type and what was attempted. A structured envelope from `failure_error_function` (and a JSON `custom_output_extractor` for normal results) lets the coordinator retry, accept partial results or annotate the gap. A gives up the manager's control and its ability to synthesize. B turns one failure into a failed brief. D does not address a model timeout.

**Q6:** **D.** Two credible sources that measure different things should both appear, with attribution, dates and methods, and the coordinator decides how to present them. A invents a number nobody published. B picks by recency, which is unrelated to correctness here. C throws away valid evidence.

**Q7:** **A.** Without a data period, a trend looks like a contradiction. A required (nullable) period on every claim, checked before conflict detection, fixes the cause. B and C silently discard valid data. D conflates publication date with data period, which is exactly the error.

**Q8:** **B.** The annotation's indices refer to the model's output text. They say "this part of the answer is supported by this URL", not "this is the page's text". Treat the span as non-verbatim and verify it against the source. A and C misread the annotation. D is not how web search citations work.

**Q9:** **D.** Citations and structured outputs cannot be combined in one request. Cite first, convert the citation blocks in code (so quotes and pointers are never retyped by the model), then extract fields in a second, structured call. A cannot fix a deterministic 400. B gives up citation guarantees. C abandons schema guarantees.

**Q10:** **C.** Coverage only says a reference exists. Precision checks that the reference supports the sentence, including its numbers. A and D measure breadth, not correctness. B relies on uncalibrated self-assessment.
</details>

<details><summary>Exercise 1 — A brief that carries all the context: reference solution</summary>

```python
def render_brief(brief: Brief, prior: list[Claim] = (), failed: list[str] = ()) -> str:
    others = [f for f in REQUIRED_FACETS if f not in brief.facets]
    if prior:
        prior_text = json.dumps([
            {"claim_id": c.claim_id, "statement": c.statement, "value": c.value, "unit": c.unit, "period": c.period,
             "evidence": [{"source_id": e.source_id, "excerpt": e.excerpt,
                           "published": REGISTRY[e.source_id].published if e.source_id in REGISTRY else None}
                          for e in c.evidence]}
            for c in prior], ensure_ascii=False)
    else:
        prior_text = "none"
    return (f'<task id="{brief.task_id}" subagent="{brief.subagent}">\n'
            f"<goal>{QUESTION}</goal>\n<as_of>{AS_OF}</as_of>\n"
            f"<your_facets>{', '.join(brief.facets)}</your_facets>\n<objective>{brief.objective}</objective>\n"
            f"<out_of_scope>{', '.join(others) or 'none'}</out_of_scope>\n"
            f"<suggested_queries>{json.dumps(brief.queries)}</suggested_queries>\n"
            f"<documents>{json.dumps(brief.documents)}</documents>\n"
            f"<prior_findings>{prior_text}</prior_findings>\n"
            f"<already_failed>{json.dumps(list(failed))}</already_failed>\n"
            f"<output_contract>{CONTRACT}</output_contract>\n</task>")
```
</details>

<details><summary>Exercise 2 — Bounded fan-out that survives failures: reference solution</summary>

```python
async def fan_out(briefs: list[Brief], worker, limit: int = 3) -> list[SubagentResult]:
    gate = asyncio.Semaphore(limit)

    async def one(b: Brief) -> SubagentResult:
        async with gate:
            try:
                return await worker(b)
            except TimeoutError as exc:
                return failed_envelope(b, "timeout", True, str(exc))
            except PermissionError as exc:
                return failed_envelope(b, "permission_denied", False, str(exc))
            except Exception as exc:  # noqa: BLE001 - one bad worker must not sink the others
                return failed_envelope(b, "internal_error", False, f"{type(exc).__name__}: {exc}")

    return list(await asyncio.gather(*(one(b) for b in briefs)))   # gather keeps order; TaskGroup would cancel siblings
```
</details>

<details><summary>Exercise 3 — Compute the envelope from the tool log: reference solution</summary>

```python
def envelope_from_log(brief: Brief, claims: list[Claim], log: list[tuple[str, str]]) -> SubagentResult:
    failed = [q for q, o in log if o not in ("hits", "empty")]
    common = dict(task_id=brief.task_id, subagent=brief.subagent, facets=brief.facets, claims=claims,
                  attempted=[q for q, _ in log], failed_queries=failed, empty_queries=[q for q, o in log if o == "empty"])
    if not failed:
        return SubagentResult(status="success" if claims else "empty", **common)
    first = next(o for q, o in log if q == failed[0])
    return SubagentResult(status="partial" if claims else "failed", failure_type=first,
                          retryable=first in ("timeout", "unavailable"),
                          coverage_impact=f"Not searched successfully: {', '.join(failed)}. "
                                          f"Findings on {', '.join(brief.facets)} may be incomplete.", **common)
```
</details>

<details><summary>Exercise 4 — Add a source-conflict rule: reference solution</summary>

```python
def classify_pair_v2(a: Claim, b: Claim) -> str:
    kind = classify_pair_v1(a, b)            # periods, agreement and quantity checks keep their order
    if kind in ("conflict", "methodology_difference") and origins(a) == origins(b):
        return "transcription_error"         # one origin, copied wrongly: not two sources disagreeing
    return kind


def conflict_record(a: Claim, b: Claim, kind: str) -> dict:
    def side(c: Claim) -> dict:
        srcs = list(dict.fromkeys(e.source_id for e in c.evidence))
        return {"claim_id": c.claim_id, "value": c.value, "unit": c.unit, "period": c.period, "sources": srcs,
                "published": [REGISTRY[s].published or "date unknown" for s in srcs],
                "methods": [REGISTRY[s].methodology or "method not stated" for s in srcs]}

    record = {"kind": kind, "claims": [a.claim_id, b.claim_id], "sides": [side(a), side(b)]}
    if kind == "transcription_error":
        origin = sorted(origins(a))[0]
        others = sorted({e.source_id for c in (a, b) for e in c.evidence} - {origin})
        record["resolution"] = f"trust {origin}; flag {', '.join(others)} for correction"
    elif kind == "conflict":
        record["resolution"] = "unresolved: the coordinator requests verification before publication"
    else:
        record["resolution"] = "explained: present both values with their methods"
    return record
```
</details>

<details><summary>Exercise 5 — Add coverage annotations to a report: reference solution</summary>

```python
def annotate_coverage(report: str, plan_facets: list[str], results: list[SubagentResult]) -> str:
    rows = {}
    for f in plan_facets:
        rs = [r for r in results if f in r.facets]
        failed = [q for r in rs for q in r.failed_queries]
        empty = [q for r in rs for q in r.empty_queries]
        if not rs:
            label, reason = "NO COVERAGE", "not searched"
        elif failed:
            has_claims = any(c.facet == f for r in rs for c in r.claims)
            label = "PARTIAL COVERAGE" if has_claims else "NO COVERAGE"
            ftype = next(r.failure_type for r in rs if r.failure_type)
            reason = f"{ftype} on " + ", ".join(f"'{q}'" for q in failed)
        else:
            label, reason = "FULL COVERAGE", ""
        rows[f] = (label, reason, empty)

    def suffix(f: str) -> str:
        label, reason, _ = rows[f]
        return f"({label}: {reason})" if reason else f"({label})"

    lines, seen = report.rstrip("\n").split("\n"), set()
    for i, line in enumerate(lines):
        m = re.match(r"^## (.+?)\s*$", line)
        if m and m.group(1).strip().lower() in rows:
            f = m.group(1).strip().lower()
            seen.add(f)
            lines[i] = f"## {m.group(1).strip()} {suffix(f)}"
    for f in plan_facets:
        if f not in seen:
            lines += ["", f"## {f.capitalize()} {suffix(f)}", "", "No findings for this facet; see Coverage and limitations."]
    lines += ["", "## Coverage and limitations", ""]
    for f in plan_facets:
        label, reason, empty = rows[f]
        line = f"- {f}: {label}" + (f" ({reason})" if reason else "")
        if empty:
            line += "; searched, no sources found: " + ", ".join(f"'{q}'" for q in empty)
        lines.append(line)
    return "\n".join(lines) + "\n"
```
</details>

<details><summary>Exercise 6 — Citation precision: reference solution</summary>

```python
def evidence_ok(c: Claim, texts: dict[str, str]) -> bool:
    for e in c.evidence:
        text = texts.get(e.source_id)
        if text is None:
            return False
        if e.verbatim and e.excerpt not in text:
            return False
        if not e.verbatim and c.value is not None and c.value not in nums(text):
            return False
    return True


def citation_precision(summary: str, claims: list[Claim], source_texts: dict[str, str]) -> float:
    by_id = {c.claim_id: c for c in claims}
    pairs = supported = 0
    for s in split_sentences(summary):
        ids = re.findall(r"\[(C\d+)\]", s)
        allowed: set[float] = set()
        for c in (by_id[i] for i in ids if i in by_id):
            allowed |= nums(c.period or "")
            for e in c.evidence:
                if e.verbatim:
                    allowed |= nums(e.excerpt)
                elif c.value is not None:
                    text = source_texts.get(e.source_id, "")
                    allowed |= set().union(*(nums(x) for x in split_sentences(text) if c.value in nums(x)))
        numbers_ok = nums(REF.sub("", s)) <= allowed
        for i in ids:
            pairs += 1
            supported += i in by_id and numbers_ok and evidence_ok(by_id[i], source_texts)
    return supported / pairs if pairs else 0.0
```
</details>